In [1]:
import os
import sys
from pathlib import Path

# Проверяем, запускался ли уже этот блок
if "IS_ROOT_SET" not in globals():
    project_root = Path.cwd().parent.parent
    os.chdir(project_root)

    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

    # Создаем флаг-индикатор
    IS_ROOT_SET = True
    print("Корневая директория инициализирована.")
else:
    print("Инициализация уже была выполнена ранее.")

Корневая директория инициализирована.


### Очистка данных

0. Импорт библиотек

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

1. Загрузка датасета (Dataset.csv)

In [3]:
data = pd.read_csv("lab5/data/raw/Dataset.csv")

In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1552210 entries, 0 to 1552209
Data columns (total 44 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   Unnamed: 0        1552210 non-null  int64  
 1   Hour              1552210 non-null  int64  
 2   HR                1398811 non-null  float64
 3   O2Sat             1349474 non-null  float64
 4   Temp              525226 non-null   float64
 5   SBP               1325945 non-null  float64
 6   MAP               1358940 non-null  float64
 7   DBP               1065656 non-null  float64
 8   Resp              1313875 non-null  float64
 9   EtCO2             57636 non-null    float64
 10  BaseExcess        84145 non-null    float64
 11  HCO3              65028 non-null    float64
 12  FiO2              129365 non-null   float64
 13  pH                107573 non-null   float64
 14  PaCO2             86301 non-null    float64
 15  SaO2              53561 non-null    float64
 16  AST        

2. Поиск и удаление дубликатов

In [5]:
duplicates = data[data.duplicated()]
duplicates

,Unnamed: 0,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID


3. Обработка пропусков

In [6]:
LAB_FEATURES = [
    "BaseExcess",
    "HCO3",
    "FiO2",
    "pH",
    "PaCO2",
    "SaO2",
    "AST",
    "BUN",
    "Alkalinephos",
    "Calcium",
    "Chloride",
    "Creatinine",
    "Glucose",
    "Lactate",
    "Magnesium",
    "Phosphate",
    "Potassium",
    "WBC",
    "Hct",
    "Hgb",
    "PTT",
    "Fibrinogen",
    "Platelets",
]
VITAL_FEATURES = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp"]
DEMO_FEATURES = ["Age", "Gender", "Unit1", "Unit2", "HospAdmTime"]


In [7]:
data[LAB_FEATURES].isna().sum()

BaseExcess      1468065
HCO3            1487182
FiO2            1422845
pH              1444637
PaCO2           1465909
SaO2            1498649
AST             1527027
BUN             1445642
Alkalinephos    1527269
Calcium         1460879
Chloride        1481744
Creatinine      1457594
Glucose         1286694
Lactate         1510764
Magnesium       1454259
Phosphate       1489909
Potassium       1407685
WBC             1452763
Hct             1414777
Hgb             1437619
PTT             1506511
Fibrinogen      1541968
Platelets       1460001
dtype: int64

In [8]:
data[VITAL_FEATURES].isna().sum()

HR        153399
O2Sat     202736
Temp     1026984
SBP       226265
MAP       193270
DBP       486554
Resp      238335
dtype: int64

Пропуски в лабораторных тестах были заполнены с использованием исторических данных пациента методом forward fill, чтобы не потерять данные о том, когда были сделаны анализы. Добавим колонки, которые будут содержать информацию в часах когда были сделаны анализы

In [9]:
def lab_features_ffil(data: pd.DataFrame) -> pd.DataFrame:
    last_measured_hour = (
        data[LAB_FEATURES]
        .notna()
        .multiply(data["ICULOS"], axis=0)
        .replace(0, np.nan)
        .groupby(data["Patient_ID"])
        .ffill()
    )

    time_since_last_lab = data["ICULOS"].values[:, None] - last_measured_hour

    time_since_last_lab.columns = [f"{col}_hours_ago" for col in LAB_FEATURES]

    data = pd.concat([data, time_since_last_lab], axis=1)
    data[LAB_FEATURES] = data.groupby("Patient_ID")[LAB_FEATURES].ffill()

    return data

In [10]:
data = lab_features_ffil(data=data)

In [11]:
data[data["Patient_ID"] == 1][["HCO3", "HCO3_hours_ago"]]

,HCO3,HCO3_hours_ago
132940,NaN,NaN
132941,NaN,NaN
132942,NaN,NaN
132943,NaN,NaN
132944,NaN,NaN
132945,NaN,NaN
132946,NaN,NaN
132947,NaN,NaN
132948,NaN,NaN
132949,NaN,NaN


Очистим жизненные показатели

In [12]:
def clean_vital_features(df: pd.DataFrame, features: list) -> pd.DataFrame:

    df = df.copy()

    vital_ranges = {
        "HR": (30, 250),  # Пульс
        "O2Sat": (50, 100),  # Сатурация
        "Temp": (30, 43),  # Температура (Цельсий)
        "SBP": (40, 280),  # Систолическое АД
        "MAP": (20, 200),  # Среднее АД
        "DBP": (20, 150),  # Диастолическое АД
        "Resp": (4, 60),  # Частота дыхания
    }

    for feature in features:
        if feature in df.columns and feature in vital_ranges:
            xmin, xmax = vital_ranges[feature]
            df.loc[(df[feature] < xmin) | (df[feature] > xmax), feature] = np.nan

    df[features] = df.groupby("Patient_ID")[features].ffill()

    return df

In [13]:
def clean_demographic_features(df: pd.DataFrame, features: list) -> pd.DataFrame:
    df = df.copy()

    if "Age" in df.columns and "Age" in features:
        df.loc[(df["Age"] <= 0) | (df["Age"] > 120), "Age"] = np.nan
        df["Age"] = df["Age"].fillna(df["Age"].median())

    if "Gender" in df.columns and "Gender" in features:
        df.loc[~df["Gender"].isin([0, 1]), "Gender"] = np.nan
        gender_mode = df["Gender"].mode()[0] if not df["Gender"].mode().empty else 0
        df["Gender"] = df["Gender"].fillna(gender_mode)

    for unit in ["Unit1", "Unit2"]:
        if unit in df.columns and unit in features:
            df[unit] = df[unit].fillna(-1)

    if "HospAdmTime" in df.columns and "HospAdmTime" in features:
        df.loc[df["HospAdmTime"] > 24, "HospAdmTime"] = np.nan
        df["HospAdmTime"] = df["HospAdmTime"].fillna(df["HospAdmTime"].median())

    return df

In [14]:
data = clean_vital_features(data, VITAL_FEATURES)
data = clean_demographic_features(data, DEMO_FEATURES)

In [15]:
data[VITAL_FEATURES].isna().sum()

HR        35482
O2Sat     38166
Temp     115187
SBP       60294
MAP       43121
DBP      320609
Resp      49525
dtype: int64

In [16]:
ENGINEERED_FEATURES = [
    "sofa_cv_score",
    "sofa_resp_score",
    "sofa_renal_score",
    "sofa_coag_score",
    "sofa_total_proxy",
    "shock_index",
    "pulse_pressure",
]


def compute_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["shock_index"] = df["HR"] / df["SBP"].replace(0, np.nan)
    df["pulse_pressure"] = df["SBP"] - df["DBP"]
    df["sofa_cv_score"] = 0
    df.loc[df["MAP"] < 70, "sofa_cv_score"] = 1

    if "FiO2" in df.columns and "O2Sat" in df.columns:
        fio2_ratio = df["FiO2"].apply(lambda x: x / 100 if x > 1.0 else x)
        fio2_ratio = fio2_ratio.replace(0, np.nan)

        spo2_fio2 = df["O2Sat"] / fio2_ratio

        df["sofa_resp_score"] = 0
        df.loc[spo2_fio2 < 400, "sofa_resp_score"] = 1
        df.loc[spo2_fio2 < 315, "sofa_resp_score"] = 2
        df.loc[spo2_fio2 < 235, "sofa_resp_score"] = 3
        df.loc[spo2_fio2 < 150, "sofa_resp_score"] = 4
    else:
        df["sofa_resp_score"] = np.nan

    if "Creatinine" in df.columns:
        df["sofa_renal_score"] = 0
        df.loc[df["Creatinine"] >= 1.2, "sofa_renal_score"] = 1
        df.loc[df["Creatinine"] >= 2.0, "sofa_renal_score"] = 2
        df.loc[df["Creatinine"] >= 3.5, "sofa_renal_score"] = 3
        df.loc[df["Creatinine"] >= 5.0, "sofa_renal_score"] = 4
    else:
        df["sofa_renal_score"] = np.nan

    if "Platelets" in df.columns:
        df["sofa_coag_score"] = 0
        df.loc[df["Platelets"] < 150, "sofa_coag_score"] = 1
        df.loc[df["Platelets"] < 100, "sofa_coag_score"] = 2
        df.loc[df["Platelets"] < 50, "sofa_coag_score"] = 3
        df.loc[df["Platelets"] < 20, "sofa_coag_score"] = 4
    else:
        df["sofa_coag_score"] = np.nan

    sofa_cols = [
        "sofa_cv_score",
        "sofa_resp_score",
        "sofa_renal_score",
        "sofa_coag_score",
    ]

    df["sofa_total_proxy"] = df[sofa_cols].sum(axis=1, min_count=1)

    return df


In [17]:
data = compute_engineered_features(data)

In [18]:
data[ENGINEERED_FEATURES].isna().sum()

sofa_cv_score            0
sofa_resp_score          0
sofa_renal_score         0
sofa_coag_score          0
sofa_total_proxy         0
shock_index          60561
pulse_pressure      320614
dtype: int64

In [19]:
def compute_advanced_rolling_features(
    df: pd.DataFrame, rolling_base: list, key_trends: list
) -> pd.DataFrame:
    df = df.copy()
    grouped = df.groupby("Patient_ID")

    for feature in rolling_base:
        if feature in df.columns:
            df[f"{feature}_mean_6h"] = (
                grouped[feature]
                .rolling(window=6, min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)
            )
            df[f"{feature}_std_6h"] = (
                grouped[feature]
                .rolling(window=6, min_periods=1)
                .std()
                .reset_index(level=0, drop=True)
                .fillna(0)
            )

    for feature in key_trends:
        if feature in df.columns:
            df[f"{feature}_diff_1h"] = grouped[feature].diff(periods=1).fillna(0)
            df[f"{feature}_diff_3h"] = grouped[feature].diff(periods=3).fillna(0)

    return df


In [20]:
KEY_TRENDS = VITAL_FEATURES + ["Lactate", "WBC", "Creatinine", "Platelets"]

ROLLING_BASE = [
    "HR",  #
    "Resp",
    "Lactate",
    "MAP",
    "O2Sat",
    "Temp",
    "SBP",
    "Glucose",
]

data = compute_advanced_rolling_features(data, ROLLING_BASE, KEY_TRENDS)

In [21]:
data.to_parquet("lab5/data/processed/processed_sepsis_data.parquet", index=False)